# LIMPIEZA DE DATOS

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/TFM_NoeliaGarciaGarcia/Pipeline'
else:
    BASE_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

import pandas as pd
import numpy as np
from scipy import stats
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import extract_features

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"BASE_PATH: {BASE_PATH}")

Entorno: Local
BASE_PATH: g:\Mi unidad\TFM_NoeliaGarciaGarcia\TFM_OxygenFlux\Pipeline


# CARGA DE DATOS

In [2]:
df_o2 = pd.read_csv(os.path.join(BASE_PATH, "DATA", "MASTER", "NM03E302DOa.dat"), sep=r"\s+")
# df_o2.head()

# Resumen rápido: tipos de datos y recuento de no-nulos
# print(df_o2.info())

In [3]:
df_temp = pd.read_csv(os.path.join(BASE_PATH, "DATA", "MASTER", "NM03E302Ta.dat"), sep=r"\s+")
# df_temp.head()

# Resumen rápido: tipos de datos y recuento de no-nulos
# print(df_temp.info())

In [4]:
# Se combina por la columna 'hour'
# Se usa 'inner' para asegurarnos de que solo queden filas que existan en ambos
df = pd.merge(df_o2, df_temp[['hour', 'temp']], on='hour', how='inner')

# Conversiones
df['O2'] = df['O2']  # umol/L == mmol/m³
df['vz'] = df['vz'] * 0.01 # cm/s -> m/s

In [5]:
# Lectura de parámetros desde config.xlsx
config = pd.read_excel(os.path.join(BASE_PATH, "NOTEBOOKS", "config.xlsx"), sheet_name="config", index_col="parameter")["value"]
fs = int(config["fs"])
window_minutes = int(config["window_minutes"])
samples_per_window = fs * 60 * window_minutes
apply_interpolation = bool(int(config.get("apply_interpolation", 1)))

# Lista de columnas a limpiar (en config como texto separado por comas: "O2,vz")
clean_columns_raw = str(config.get("clean_columns", "O2,vz"))
clean_columns = tuple(c.strip() for c in clean_columns_raw.split(",") if c.strip())

print(f"fs={fs}, window_minutes={window_minutes}, samples_per_window={samples_per_window}")
print(f"apply_interpolation={apply_interpolation}")
print(f"clean_columns={clean_columns}")

fs=8, window_minutes=15, samples_per_window=7200
apply_interpolation=True
clean_columns=('O2', 'vz')


# PARAMETRIZACIÓN

In [6]:
# PARÁMETROS (los que NO están en config.xlsx)
max_sample_normality = 5000 # Sirve para no pasar demasiadas muestras al test de Shapiro. Con ventanas de 7200 muestras, el test de normalidad puede ser demasiado sensible.
random_state = 0 # Semilla generador aleatorio
near_constant_std = 1e-10 # Sirve para detectar ventanas casi constantes. Por ejemplo, si una señal apenas cambia dentro de una ventana, no tiene sentido calcular asimetría, curtosis o normalidad.

min_valid_samples = 30

max_gap_seconds = 2 # Máximo de segundos que se permite interpolar datos
z_thresh = 6.0 # z-score clásico
min_valid_fraction = 0.95  # Porcentaje mínimo que se usa para validar si una ventana es válida. Es el total de filas que tienen o2 y vz válidos.

# ESTADÍSTICAS

In [7]:
TSFRESH_STATS_PARAMETERS = {
    "mean": None,
    "median": None,
    "standard_deviation": None,
    "variance": None,
    "minimum": None,
    "maximum": None,
    "absolute_maximum": None,
    "sum_values": None,
    "root_mean_square": None,
    "mean_abs_change": None,
    "mean_change": None,
    "skewness": None,
    "kurtosis": None,
    "quantile": [
        {"q": 0.01},
        {"q": 0.05},
        {"q": 0.25},
        {"q": 0.75},
        {"q": 0.95},
        {"q": 0.99},
    ],
}

In [8]:
def extract_tsfresh_features_one_window(
    df_window: pd.DataFrame,
    relevant_cols=("O2", "vz"),
    default_fc_parameters=TSFRESH_STATS_PARAMETERS,
    n_jobs=0
):
    """
    Extrae estadísticas de una única ventana usando tsfresh.

    Entrada:
        df_window: dataframe correspondiente a una sola ventana.

    Devuelve:
        df_features: dataframe de una fila con las estadísticas de esa ventana.
    """

    dfp = df_window.copy().reset_index(drop=True)

    relevant_cols = [col for col in relevant_cols if col in dfp.columns]

    if len(relevant_cols) == 0:
        raise ValueError("Ninguna de las columnas indicadas existe en el dataframe.")

    for col in relevant_cols:
        dfp[col] = pd.to_numeric(dfp[col], errors="coerce")

    if not np.isfinite(dfp[relevant_cols].to_numpy(dtype=float)).all():
        raise ValueError(
            "La ventana contiene NaN o infinitos. "
            "Limpia la ventana antes de calcular estadísticas con tsfresh."
        )

    dfp["window_id"] = 0
    dfp["time_in_window"] = np.arange(len(dfp))

    df_tsfresh_input = dfp[
        ["window_id", "time_in_window", *relevant_cols]
    ].copy()

    df_features = extract_features(
        df_tsfresh_input,
        column_id="window_id",
        column_sort="time_in_window",
        default_fc_parameters=default_fc_parameters,
        disable_progressbar=True,
        n_jobs=n_jobs
    )

    impute(df_features)

    df_features.index.name = "window_id"
    df_features = df_features.reset_index()

    return df_features

In [9]:
def extract_tsfresh_features_by_window(
    df: pd.DataFrame,
    relevant_cols=("O2", "vz"),
    default_fc_parameters=TSFRESH_STATS_PARAMETERS,
    n_jobs=0
):
    """
    Divide el dataframe en ventanas y extrae estadísticas de cada ventana
    usando tsfresh.

    Asume que los datos ya están limpios.
    """

    dfp = df.copy().sort_values("hour").reset_index(drop=True)

    relevant_cols = [col for col in relevant_cols if col in dfp.columns]

    if len(relevant_cols) == 0:
        raise ValueError("Ninguna de las columnas indicadas existe en el dataframe.")

    for col in relevant_cols:
        dfp[col] = pd.to_numeric(dfp[col], errors="coerce")

    if not np.isfinite(dfp[relevant_cols].to_numpy(dtype=float)).all():
        raise ValueError(
            "El dataframe contiene NaN o infinitos. "
            "Limpia los datos antes de calcular estadísticas con tsfresh."
        )

    samples_per_window = int(fs * 60 * window_minutes)
    n_windows = len(dfp) // samples_per_window

    if n_windows == 0:
        return pd.DataFrame()

    limit = n_windows * samples_per_window
    dfp = dfp.iloc[:limit].copy()

    dfp["window_id"] = np.repeat(
        np.arange(n_windows),
        samples_per_window
    )

    dfp["time_in_window"] = np.tile(
        np.arange(samples_per_window),
        n_windows
    )

    df_tsfresh_input = dfp[
        ["window_id", "time_in_window", *relevant_cols]
    ].copy()

    df_features = extract_features(
        df_tsfresh_input,
        column_id="window_id",
        column_sort="time_in_window",
        default_fc_parameters=default_fc_parameters,
        disable_progressbar=True,
        n_jobs=n_jobs
    )

    impute(df_features)

    df_features.index.name = "window_id"
    df_features = df_features.reset_index()

    return df_features

In [10]:
tsfresh_features_diag = extract_tsfresh_features_by_window(
    df=df,
    relevant_cols=clean_columns,
    default_fc_parameters=TSFRESH_STATS_PARAMETERS,
    n_jobs=0
)

# display(df_features.head())

In [11]:
def obtener_estadistica_tsfresh(
    df_features: pd.DataFrame,
    window_id: int,
    variable: str,
    estadistica: str,
    q=None
):
    """
    Recupera una estadística calculada por tsfresh para una ventana concreta.

    Ejemplos de columnas:
        O2__mean
        O2__median
        O2__standard_deviation
        O2__minimum
        O2__maximum
        O2__skewness
        O2__kurtosis
        O2__quantile__q_0.25
    """

    if df_features is None or df_features.empty:
        return np.nan

    if estadistica == "quantile":
        col = f"{variable}__quantile__q_{q}"
    else:
        col = f"{variable}__{estadistica}"

    if col not in df_features.columns:
        return np.nan

    value = df_features.loc[
        df_features["window_id"] == window_id,
        col
    ]

    if value.empty:
        return np.nan

    return float(value.iloc[0])

# DIAGNÓSTICO

In [12]:
def diagnosticar_ventanas(
    df,
    cols = ("O2", "vz")
):
    df_diag = df.copy().sort_values("hour").reset_index(drop=True)

    n_windows = len(df_diag) // samples_per_window
    limit = n_windows * samples_per_window

    df_diag = df_diag.iloc[:limit].copy() # Si sobran muestras al final que no forman una ventana completa, se eliminan para este diagnóstico
    df_diag["window_id"] = np.repeat(np.arange(n_windows), samples_per_window)

    rng = np.random.default_rng(random_state) # Para shapiro no se pueden tener más de 5000 muestras, por lo que se usan siempre las mismas usando siempre la misma semilla
    rows = []

    for wid, g in df_diag.groupby("window_id", sort=False):
        for col in cols:
            if col not in g.columns:
                continue

            x = pd.to_numeric(g[col], errors="coerce").to_numpy(dtype=float) # Esto convierte la columna a números y elimina valores no válidos
            x = x[np.isfinite(x)]

            if len(x) < min_valid_samples: # Si después de eso quedan pocos datos la función dice que para esa ventana hay insuficientes datos
                rows.append({
                    "window_id": wid,
                    "variable": col,
                    "n": len(x),
                    "metodo_recomendado": "insuficientes_datos"
                })
                continue

            # Estadísticas calculadas con tsfresh
            mean = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "mean"
            )

            std = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "standard_deviation"
            )

            variance = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "variance"
            )

            median = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "median"
            )

            q1 = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "quantile", q=0.25
            )

            q3 = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "quantile", q=0.75
            )

            x_min = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "minimum"
            )

            x_max = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "maximum"
            )

            skew = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "skewness"
            )

            kurt = obtener_estadistica_tsfresh(
                tsfresh_features_diag, wid, col, "kurtosis"
            )

            # Derivadas de estadísticas de tsfresh
            iqr = q3 - q1
            data_range = x_max - x_min

            # Porcentajes
            frac_unique = len(np.unique(x)) / len(x)
            frac_zeros = np.mean(x == 0)

            mad = np.median(np.abs(x - median))

            # Caso importante: calcular si la ventana es constante o casi constante
            is_constant = data_range == 0
            is_near_constant = std < near_constant_std or iqr == 0

            if is_constant or is_near_constant: # Si es constante o casi constante no tiene sentido aplicar tests de normalidad, ni skew o kurtosis. Por eso para esa ventana se devuelve que hay que aplicar reglas físicas.
                rows.append({
                    "window_id": wid,
                    "variable": col,
                    "n": len(x),
                    "mean": mean,
                    "std": std,
                    "median": median,
                    "q1": q1,
                    "q3": q3,
                    "iqr": iqr,
                    "mad": mad,
                    "min": x_min,
                    "max": x_max,
                    "range": data_range,
                    "skew": np.nan,
                    "kurtosis_excess": np.nan,
                    "shapiro_p": np.nan,
                    "anderson_stat": np.nan,
                    "anderson_reject_5": np.nan,
                    "frac_unique": frac_unique,
                    "frac_zeros": frac_zeros,
                    "is_constant": is_constant,
                    "is_near_constant": is_near_constant,
                    "metodo_recomendado": "reglas_fisicas_o_sensor_constante"
                })
                continue

            # Solo calculamos asimetría y curtosis si la ventana tiene variabilidad suficiente
            skew = stats.skew(x, bias=False)
            kurt = stats.kurtosis(x, fisher=True, bias=False)

            if len(x) > max_sample_normality:
                x_test = rng.choice(x, size=max_sample_normality, replace=False)
            else:
                x_test = x

            # Se aplica shapiro y anderson para comprobar si la ventana se parece a una normal
            try:
                shapiro_stat, shapiro_p = stats.shapiro(x_test)
            except Exception:
                shapiro_p = np.nan

            try:
                anderson = stats.anderson(x_test, dist="norm")
                anderson_stat = anderson.statistic
                anderson_crit_5 = anderson.critical_values[2]
                anderson_reject_5 = anderson_stat > anderson_crit_5
            except Exception:
                anderson_stat = np.nan
                anderson_reject_5 = np.nan

            # Variables para decicir si usar tukey, mad o iqr
            approx_symmetric = abs(skew) < 0.5
            tails_not_extreme = abs(kurt) < 3
            shapiro_ok = shapiro_p > 0.01 if np.isfinite(shapiro_p) else False
            mad_usable = mad > 0
            iqr_usable = iqr > 0

            if not iqr_usable and not mad_usable:
                metodo = "reglas_fisicas_o_sensor_constante"
            elif frac_zeros > 0.3 or frac_unique < 0.05:
                metodo = "reglas_fisicas_o_percentiles"
            elif approx_symmetric and tails_not_extreme and shapiro_ok:
                metodo = "tukey"
            elif mad_usable:
                metodo = "mad_hampel"
            else:
                metodo = "percentiles_robustos"

            rows.append({
                "window_id": wid,
                "variable": col,
                "n": len(x),
                "mean": mean,
                "std": std,
                "median": median,
                "q1": q1,
                "q3": q3,
                "iqr": iqr,
                "mad": mad,
                "min": x_min,
                "max": x_max,
                "range": data_range,
                "skew": skew,
                "kurtosis_excess": kurt,
                "shapiro_p": shapiro_p,
                "anderson_stat": anderson_stat,
                "anderson_reject_5": anderson_reject_5,
                "frac_unique": frac_unique,
                "frac_zeros": frac_zeros,
                "is_constant": is_constant,
                "is_near_constant": is_near_constant,
                "metodo_recomendado": metodo
            })

    return pd.DataFrame(rows)

In [13]:
diag = diagnosticar_ventanas(
    df=df,
    cols=clean_columns
)

# display(diag.groupby(["variable", "metodo_recomendado"]).size())

metodo_por_variable = (
    diag
    .groupby("variable")["metodo_recomendado"]
    .agg(lambda s: s.value_counts().idxmax())
    .reset_index()
)

display(metodo_por_variable)

,variable,metodo_recomendado
0,O2,mad_hampel
1,vz,mad_hampel


# IMPLEMENTACIÓN

La limpieza se aplicó únicamente sobre las variables utilizadas en el cálculo del flujo, O2 y vz. Se realizó por ventanas temporales para no mezclar distribuciones de distintos periodos. Se eligió MAD/Hampel porque el diagnóstico por ventana no justificaba asumir normalidad ni aplicar Tukey de forma general. Además, se interpolaron solo huecos cortos y se marcaron ventanas no válidas cuando la proporción de datos útiles era insuficiente.

In [14]:
def outliers_mad_hampel(x): # Esta función recibe una señal de una ventana, por ejemplo los 7200 valores de O2 o vz, y devuelve una máscara booleana
    x = np.asarray(x, dtype=float)

    bad = ~np.isfinite(x) # Detecta valores válidos. True si es un número, False si es NaN, inf... Se invierte con ~
    valid = x[~bad] # Crea un array con los valores que se pueden usar

    if len(valid) < 30:  # Evitar muestras muy pequeñas
        return bad

    med = np.nanmedian(valid) # Calcular mediana
    mad = np.nanmedian(np.abs(valid - med)) # Calcular MAD (Median Absolute Deviation)

    if mad == 0: # Si MAD es cero, no puede calcular outliers
        return bad

    # Esto mide cuánto se aleja cada punto de la mediana, pero usando MAD en vez de desviación estándar.
    # El factor 0.6745 se usa para que el resultado sea comparable a un z-score clásico cuando la distribución es aproximadamente normal.
    robust_z = 0.6745 * (x - med) / mad

    # Marcar outliers
    bad |= np.abs(robust_z) > z_thresh

    return bad


In [15]:
def es_senal_plana_o_invalida(x):
    x = np.asarray(x, dtype=float)
    x_valid = x[np.isfinite(x)]

    # Si no hay suficientes datos válidos, la ventana no es fiable
    if len(x_valid) < min_valid_samples:
        return True

    # Si la desviación típica es casi cero, la señal está plana
    return np.std(x_valid) < near_constant_std

In [16]:
def limpiar_ventana(
    df,
    cols
):
    dfc = df.copy().sort_values("hour").reset_index(drop=True)

    # Usar solo columnas solicitadas que existan en el dataframe
    cols = tuple(c for c in cols if c in dfc.columns)
    if len(cols) == 0:
        raise ValueError("No hay columnas válidas en 'cols' para limpiar.")

    n_windows = len(dfc) // samples_per_window
    limit = n_windows * samples_per_window

    dfc = dfc.iloc[:limit].copy()
    dfc["window_id"] = np.repeat(np.arange(n_windows), samples_per_window)

    # Columnas de control (dinámicas según cols)
    for col in cols:
        dfc[f"outlier_{col}"] = False
        dfc[f"interpolated_{col}"] = False

    # Máximo hueco interpolable
    # Un pico aislado se puede interpolar, pero un fallo largo de sensor no conviene inventarlo.
    # Por eso solo se interpolan huecos de hasta 16 muestras, es decir, hasta 2 segundos.
    max_gap_samples = int(max_gap_seconds * fs)

    qc_rows = []

    # Recorrer cada ventana
    for wid, g in dfc.groupby("window_id", sort=False):
        idx = g.index.to_numpy()

        n_outliers_by_col = {col: 0 for col in cols}
        n_interp_by_col = {col: 0 for col in cols}
        flat_by_col = {col: False for col in cols}

        for col in cols:
            x = pd.to_numeric(dfc.loc[idx, col], errors="coerce").to_numpy(dtype=float)

            # Guardar valores originales antes de tocar nada
            x_original = x.copy()

            # Valores no finitos (NaN, inf) que ya venían mal de origen
            bad_original = ~np.isfinite(x)

            # Concentración de oxígeno negativa no es válida
            if col == "O2":
                bad_original |= x < 0

            # Outliers robustos por ventana usando la fórmula
            bad_outlier = outliers_mad_hampel(x)
            # Quitar los que ya eran bad_original para no contar doble
            is_outlier = bad_outlier & ~bad_original

            # Se marcan los puntos outlier en la columna de control
            dfc.loc[idx[is_outlier], f"outlier_{col}"] = True
            n_outliers_by_col[col] = int(is_outlier.sum())

            # INTERPOLACIÓN (controlada por config: apply_interpolation)
            if apply_interpolation:
                # Se ponen a NaN temporalmente para interpolar
                bad_all = bad_original | is_outlier
                dfc.loc[idx[bad_all], col] = np.nan

                before = dfc.loc[idx, col].isna()

                dfc.loc[idx, col] = (
                    dfc.loc[idx, col]
                    .interpolate(
                        method="linear", # Esto rellena huecos pequeños usando interpolación lineal.
                        limit=max_gap_samples,
                        limit_direction="both"
                    )
                )

                after = dfc.loc[idx, col].isna()
                interpolated = before & ~after

                # Restaurar valores originales donde la interpolación no pudo rellenar
                # (así no dejamos NaN residuales en la señal limpia)
                still_nan = dfc.loc[idx, col].isna()
                restore_mask = still_nan
                restore_positions = idx[restore_mask.to_numpy()]
                dfc.loc[restore_positions, col] = x_original[restore_mask.to_numpy()]
            else:
                interpolated = np.zeros(len(idx), dtype=bool)

            # Marcar puntos interpolados
            dfc.loc[idx, f"interpolated_{col}"] = np.asarray(interpolated, dtype=bool)
            n_interp_by_col[col] = int(np.asarray(interpolated, dtype=bool).sum())

            # Señal plana por columna
            flat_by_col[col] = es_senal_plana_o_invalida(
                dfc.loc[idx, col].to_numpy(dtype=float)
            )

        # ¿Tiene outliers en esta ventana?
        has_outliers = sum(n_outliers_by_col.values()) > 0

        # Guardar resumen de la ventana
        row = {
            "window_id": wid,
            "hour_start": dfc.loc[idx, "hour"].iloc[0],
            "hour_end": dfc.loc[idx, "hour"].iloc[-1],
            "hour_mid": dfc.loc[idx, "hour"].mean(),
            "has_outliers": has_outliers,
        }

        for col in cols:
            row[f"n_outliers_{col}"] = n_outliers_by_col[col]
            row[f"n_interpolated_{col}"] = n_interp_by_col[col]
            row[f"flat_{col}"] = flat_by_col[col]

        qc_rows.append(row)

    qc_windows = pd.DataFrame(qc_rows)

    return dfc, qc_windows

In [17]:
df_limpio, qc_windows = limpiar_ventana(
    df=df,
    cols=clean_columns
)

In [18]:
df_limpio.head()

,hour,vx,vy,vz,O2,pres,temp,window_id,outlier_O2,interpolated_O2,outlier_vz,interpolated_vz
0,13.000026,-2.110,2.385,-0.00510,218.730,1475.35,14.8686,0,False,False,False,False
1,13.000061,-2.280,2.515,-0.00540,218.835,1475.35,14.8692,0,False,False,False,False
2,13.000095,-2.990,2.710,-0.00685,218.597,1475.35,14.8692,0,False,False,False,False
3,13.000130,-2.945,2.895,-0.00685,218.746,1477.60,14.8692,0,False,False,False,False
4,13.000165,-3.045,2.745,-0.00560,218.348,1473.05,14.8692,0,False,False,False,False


In [19]:
summary_cols = [
    c for c in qc_windows.columns
    if c.startswith("n_outliers_") or c.startswith("n_interpolated_")
]

if "has_outliers" in qc_windows.columns:
    summary_cols.append("has_outliers")

resumen_limpieza = qc_windows[summary_cols].sum()

display(resumen_limpieza)

n_outliers_O2        10614
n_interpolated_O2     3985
n_outliers_vz         3654
n_interpolated_vz     3518
has_outliers           335
dtype: int64

In [20]:
for col in clean_columns:
    out_col = f"outlier_{col}"
    if out_col in df_limpio.columns:
        print(f"Porcentaje outliers {col}:", 100 * df_limpio[out_col].mean())

print("Porcentaje ventanas con outliers:", 100 * qc_windows["has_outliers"].mean())

Porcentaje outliers O2: 0.21364734299516908
Porcentaje outliers vz: 0.07355072463768116
Porcentaje ventanas con outliers: 48.55072463768116


In [21]:
tsfresh_features = extract_tsfresh_features_by_window(
    df=df_limpio,
    relevant_cols=("O2", "vz", "vx", "vy", "temp", "pres"),
    default_fc_parameters=TSFRESH_STATS_PARAMETERS,
    n_jobs=0
)

print(f"Features calculadas: {tsfresh_features.shape}")

Features calculadas: (690, 115)


# PROCESAMIENTO Y CREACIÓN DE RAW DATA



In [22]:
output_path = os.path.join(BASE_PATH, "DATA", "RAW", "df_master.csv")
df.to_csv(output_path, index=False)
print(f"Datos guardados exitosamente en: {output_path}")

Datos guardados exitosamente en: g:\Mi unidad\TFM_NoeliaGarciaGarcia\TFM_OxygenFlux\Pipeline\DATA\RAW\df_master.csv


In [23]:
output_path = os.path.join(BASE_PATH, "DATA", "CLEAN", "df_clean.csv")
df_limpio.to_csv(output_path, index=False)
print(f"Datos guardados exitosamente en: {output_path}")

Datos guardados exitosamente en: g:\Mi unidad\TFM_NoeliaGarciaGarcia\TFM_OxygenFlux\Pipeline\DATA\CLEAN\df_clean.csv


In [24]:
output_path = os.path.join(BASE_PATH, "DATA", "CLEAN", "qc.csv")
qc_windows.to_csv(output_path, index=False)
print(f"Datos guardados exitosamente en: {output_path}")

Datos guardados exitosamente en: g:\Mi unidad\TFM_NoeliaGarciaGarcia\TFM_OxygenFlux\Pipeline\DATA\CLEAN\qc.csv


In [27]:
output_path = os.path.join(BASE_PATH, "DATA", "PROCESSED", "df_features.csv")
tsfresh_features.to_csv(output_path, index=False)
print(f"Datos guardados exitosamente en: {output_path}")

Datos guardados exitosamente en: g:\Mi unidad\TFM_NoeliaGarciaGarcia\TFM_OxygenFlux\Pipeline\DATA\PROCESSED\df_features.csv
